In [6]:
import cv2
import numpy as np

img_a = cv2.imread(r"E:\JupyterProject1\a.jpg")
img_b = cv2.imread(r"E:\JupyterProject1\b.jpg")

gray_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
gray_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp_a, des_a = sift.detectAndCompute(gray_a, None)
kp_b, des_b = sift.detectAndCompute(gray_b, None)

FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des_a, des_b, k=2)

good_matches = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good_matches.append(m)

src_pts = np.float32([kp_a[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp_b[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
H, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

h_a, w_a = img_a.shape[:2]
h_b, w_b = img_b.shape[:2]

pts_a = np.float32([[0, 0], [0, h_a], [w_a, h_a], [w_a, 0]]).reshape(-1, 1, 2)
dst_corners = cv2.perspectiveTransform(pts_a, H)

all_corners = np.concatenate([dst_corners, np.float32([[0, 0], [0, h_b], [w_b, h_b], [w_b, 0]]).reshape(-1, 1, 2)], axis=0)
[x_min, y_min] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
[x_max, y_max] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

translation_matrix = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float32)

stitched_img = cv2.warpPerspective(img_a, translation_matrix @ H, (x_max - x_min, y_max - y_min))

stitched_img[-y_min : h_b - y_min, -x_min : w_b - x_min] = img_b

cv2.namedWindow('Stitched Result', cv2.WINDOW_NORMAL)
cv2.imshow('Stitched Result', stitched_img)
cv2.waitKey(0)
cv2.destroyAllWindows()